# BERT Sentiment Analysis (real Transformers run)

Offline baseline + real Transformers inference using a tiny model to keep downloads small.

_Last rebuild: **2026-02-16 03:21:21**_

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

np.random.seed(42)


In [2]:
pos = ['I love this product', 'Excellent quality', 'Amazing service', 'Works perfectly']
neg = ['Terrible experience', 'Waste of money', 'Very disappointed', 'Broke immediately']

texts = [s for s in pos for _ in range(80)] + [s for s in neg for _ in range(80)]
y = np.array([1]*(len(pos)*80) + [0]*(len(neg)*80))

df = pd.DataFrame({'text': texts, 'label': y})
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.25, random_state=42, stratify=df['label'])

baseline = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2))), ('clf', LogisticRegression(max_iter=200))])
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)
print('TFIDF baseline')
print(classification_report(y_test, pred))

TFIDF baseline
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        80
           1       1.00      1.00      1.00        80

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160



In [3]:
# Real transformers inference (standard model, downloads weights on first run)
from transformers import pipeline

model_id = 'distilbert-base-uncased-finetuned-sst-2-english'

# If a previous partial download corrupted the cache, remove and retry once.
try:
    clf = pipeline('sentiment-analysis', model=model_id)
except Exception as e:
    msg = str(e).lower()
    if 'state dictionary' in msg and 'corrupted' in msg:
        import shutil
        from pathlib import Path
        cache_root = Path.home() / '.cache' / 'huggingface'
        print('Detected corrupted HF cache, removing:', cache_root)
        shutil.rmtree(cache_root, ignore_errors=True)
        clf = pipeline('sentiment-analysis', model=model_id)
    else:
        raise

print(clf('I absolutely love this, it is fantastic'))
print(clf('This is horrible, I want a refund'))

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998804330825806}]
[{'label': 'NEGATIVE', 'score': 0.9997491240501404}]


In [4]:
print('DONE 2026-02-16 03:21:21')

DONE 2026-02-16 03:21:21
